In [ ]:
import os, re, cv2, glob, time, difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime

# ---------- EDIT THESE PATHS ----------
IMAGES_FOLDER   = "images"                 # folder with car1.jpg, car2.jpg ...
PLATE_WEIGHTS   = "weights/best.pt"        # your trained YOLO plate detector
GROUND_TRUTH_CSV = "ground_truth.csv"      # columns: Image Name, Actual Plate
# --------------------------------------

WATERMARKS = {"ALAMY", "SHUTTERSTOCK", "ISTOCK", "GETTY", "DREAMSTIME", "ADOBE", "STOCK"}
MIN_TEXT_CONF = 0.40
MAX_PLATES_PER_IMAGE = 3

os.makedirs("output", exist_ok=True)
os.makedirs("database", exist_ok=True)
print("OpenCV", cv2.__version__)

In [ ]:
import re

STATE_CODES = set("AN AP AR AS BR CG CH DD DL DN GA GJ HP HR JH JK KA KL LA LD MH ML MN MP MZ NL OD OR PB PY RJ SK TN TR TS UK UP WB".split())
TO_DIGIT  = {'O':'0','Q':'0','D':'0','U':'0','I':'1','L':'1','Z':'2','S':'5','G':'6','T':'7','B':'8','A':'4'}
TO_LETTER = {'0':'D','1':'I','2':'Z','4':'A','5':'S','6':'G','7':'T','8':'B'}
HEAD_ALTS = {'0':'OD','1':'IL','2':'ZB','4':'A','5':'S','6':'G','7':'T','8':'BW'}   # a digit in the state-code slot
MAX_COST = 4.5      # above this the text is NOT treated as an Indian plate (stops garbage like AR12ONA9319)

def _conv(chars, want_digit):
    out, cost = "", 0
    for ch in chars:
        if want_digit:
            if ch.isdigit(): out += ch
            elif ch in TO_DIGIT: out += TO_DIGIT[ch]; cost += 1
            else: return None
        else:
            if ch.isalpha(): out += ch
            elif ch in TO_LETTER: out += TO_LETTER[ch]; cost += 1
            else: return None
    return out, cost

def _head(chars):
    opts = [("", 0)]
    for ch in chars:
        if ch.isalpha(): c = [(ch, 0)]
        elif ch in HEAD_ALTS: c = [(a, 1) for a in HEAD_ALTS[ch]]
        else: return None
        opts = [(o + a, k + b) for o, k in opts for a, b in c]
    ok = [o for o in opts if o[0] in STATE_CODES]
    return min(ok, key=lambda x: x[1]) if ok else None

def fit_indian(s):
    """SS DD[D] L{0-3} DDDD (state, district, series, number) or Bharat series YY BH DDDD L{1,2}."""
    if not 8 <= len(s) <= 11:
        return None
    if re.fullmatch(r'\d{2}BH\d{4}[A-Z]{1,2}', s):
        return s, 0
    h, t = _head(s[:2]), _conv(s[-4:], True)
    if h is None or t is None:
        return None
    mid, best = s[2:-4], None
    for d in (2, 1):
        ser = mid[d:]
        if len(mid) < d or len(ser) > 3 or (len(ser) == 0 and d == 1):
            continue
        dist, sr = _conv(mid[:d], True), _conv(ser, False)
        if dist is None or sr is None:
            continue
        cost = h[1] + t[1] + dist[1] + sr[1] + (0.3 if d == 1 else 0) + (0.3 if len(ser) == 3 else 0) + (1.0 if len(ser) == 0 else 0)
        cand = (h[0] + dist[0] + sr[0] + t[0], cost)
        if best is None or cand[1] < best[1]:
            best = cand
    return best

def best_plate(raw):
    s = re.sub(r'[^A-Z0-9]', '', str(raw).upper())
    if re.fullmatch(r'[A-Z]{2}\d{4}', s) and s[:2] in STATE_CODES:      # old short format, e.g. AR6357
        return s, True
    best = None
    for L in range(min(len(s), 11), 7, -1):
        for i in range(len(s) - L + 1):
            r = fit_indian(s[i:i+L])
            if r:
                cost = r[1] + 1.5 * (len(s) - L)
                if best is None or cost < best[1]:
                    best = (r[0], cost)
    if best and best[1] <= MAX_COST:
        return best[0], True
    return s, False


# quick self-test
assert best_plate("MH200V2363") == ("MH20DV2363", True)
assert best_plate("KL11803333") == ("KL11BD3333", True)
assert best_plate("AR6357") == ("AR6357", True)
print("format fixer OK")

In [ ]:
from ultralytics import YOLO
from paddleocr import PaddleOCR

assert os.path.exists(PLATE_WEIGHTS), f"Weights not found: {PLATE_WEIGHTS}"
plate_model = YOLO(PLATE_WEIGHTS)

ocr = PaddleOCR(
    use_angle_cls=True, lang="en", show_log=False,
    det_db_thresh=0.2,          # more sensitive text detector (finds small 2nd row)
    det_db_box_thresh=0.4,
    det_db_unclip_ratio=2.0,
)
print("Models ready")

In [ ]:
# 2) Plate detection cascade: tries harder only when the previous stage found nothing
from ultralytics import YOLO
vehicle_model = YOLO("yolov8n.pt")            # COCO model, downloads automatically
VEHICLE_CLASSES = [2, 3, 5, 7]                # car, motorcycle, bus, truck


def _predict(img, conf, imgsz, augment=False, dx=0, dy=0):
    res = plate_model.predict(source=img, conf=conf, iou=0.45, imgsz=imgsz,
                              augment=augment, verbose=False)[0]
    out = []
    for b in res.boxes:
        x1, y1, x2, y2 = map(int, b.xyxy[0])
        out.append((x1 + dx, y1 + dy, x2 + dx, y2 + dy, float(b.conf[0])))
    return out


def _plate_shaped(b):
    w, h = b[2] - b[0], b[3] - b[1]
    return w >= 20 and h >= 8 and 0.9 <= w / h <= 7.0       # single-row ~4:1, two-row ~1.5:1


def detect_plates(image):
    H, W = image.shape[:2]
    stages = []

    stages.append(lambda: _predict(image, 0.10, 1280, augment=True))      # stage 1: normal
    stages.append(lambda: _predict(image, 0.04, 1920))                    # stage 2: bigger + very low conf

    def by_vehicle():                                                     # stage 3: look inside each vehicle
        found = []
        vr = vehicle_model.predict(source=image, conf=0.25, classes=VEHICLE_CLASSES, verbose=False)[0]
        vb = sorted([tuple(map(int, b.xyxy[0])) for b in vr.boxes],
                    key=lambda t: -(t[2]-t[0]) * (t[3]-t[1]))[:3]
        for x1, y1, x2, y2 in vb:
            top = y1 + int(0.35 * (y2 - y1))                              # plates sit in the lower part
            sub = image[top:y2, x1:x2]
            if sub.size:
                found += _predict(sub, 0.04, 1280, dx=x1, dy=top)
        return found
    stages.append(by_vehicle)

    def by_tiles():                                                       # stage 4: 2x2 overlapping tiles
        found = []
        tw, th = int(W * 0.6), int(H * 0.6)
        for x0 in (0, W - tw):
            for y0 in (0, H - th):
                found += _predict(image[y0:y0+th, x0:x0+tw], 0.04, 1280, dx=x0, dy=y0)
        return found
    stages.append(by_tiles)

    for stage in stages:
        try:
            boxes = [b for b in stage() if _plate_shaped(b)]
        except Exception as e:
            print("detector stage error:", e); boxes = []
        if boxes:
            boxes.sort(key=lambda t: -t[4])
            kept = []
            for b in boxes:                       # simple de-duplication of overlapping boxes
                if all(_iou(b, k) < 0.4 for k in kept):
                    kept.append(b)
            top = kept[0][4]
            return [b for b in kept if b[4] >= 0.35 * top][:MAX_PLATES_PER_IMAGE]
    return []


def _iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua else 0

In [ ]:
# 3) Crops: several vertical paddings, because the detector box often covers only the top row
def crop_box(image, box, px_ratio, py_ratio):
    x1, y1, x2, y2, _ = box
    H, W = image.shape[:2]
    px, py = int((x2 - x1) * px_ratio), int((y2 - y1) * py_ratio)
    return image[max(0, y1-py):min(H, y2+py), max(0, x1-px):min(W, x2+px)]


def crop_options(image, box):
    w, h = box[2] - box[0], box[3] - box[1]
    two_row_like = (w / max(h, 1)) < 3.0
    opts = [(0.08, 0.15), (0.10, 0.45)]               # normal, tall
    if two_row_like:
        opts.append((0.15, 0.90))                     # very tall: box probably sits on the top row only
    return [crop_box(image, box, px, py) for px, py in opts], two_row_like

In [ ]:
def ocr_lines(img):
    """Run PaddleOCR (2.x API) and return [(cx, cy, height, text, conf)] after filtering."""
    out = ocr.ocr(img, cls=True)
    if not out or not out[0]:
        return []
    lines = []
    for box, (text, conf) in out[0]:
        t = re.sub(r"[^A-Za-z0-9]", "", text).upper()
        if conf < MIN_TEXT_CONF or len(t) == 0 or t in WATERMARKS:
            continue
        xs = [p[0] for p in box]; ys = [p[1] for p in box]
        lines.append((np.mean(xs), np.mean(ys), max(ys) - min(ys), t, conf))
    return lines


def join_rows(lines):
    """Group text boxes into rows (handles 2-row plates), read each row left->right."""
    if not lines:
        return "", 0.0
    lines = sorted(lines, key=lambda l: l[1])
    med_h = np.median([l[2] for l in lines])
    rows, cur = [], [lines[0]]
    for l in lines[1:]:
        if abs(l[1] - np.mean([c[1] for c in cur])) < 0.6 * med_h:
            cur.append(l)
        else:
            rows.append(cur); cur = [l]
    rows.append(cur)
    text = "".join("".join(c[3] for c in sorted(r, key=lambda c: c[0])) for r in rows)
    conf = float(np.mean([l[4] for l in lines]))
    return text, conf


def ocr_rec_only(img):
    """Recognition only (no text detection) - for an already tight single text row."""
    try:
        out = ocr.ocr(img, det=False, cls=False)
        item = out[0] if out else None
        if item and isinstance(item[0], (tuple, list)) and isinstance(item[0][0], str):
            text, conf = item[0]
        elif item and isinstance(item[0], str):
            text, conf = item
        else:
            return "", 0.0
        return re.sub(r"[^A-Z0-9]", "", text.upper()), float(conf)
    except Exception:
        return "", 0.0

In [ ]:
# PATCH A - more image variants = more votes (fixes single-letter confusions like H/B, 3/2)
def make_variants(img, target_h=160):
    h = img.shape[0]
    def prep(th):
        s = float(np.clip(th / max(h, 1), 1.0, 4.0)) if th > 0 else 1.0
        return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC) if s > 1.01 else img.copy()
    big = prep(target_h)
    gray = cv2.cvtColor(big, cv2.COLOR_BGR2GRAY)
    out = {
        "color": big,
        "clahe": cv2.cvtColor(cv2.createCLAHE(2.0, (4, 4)).apply(gray), cv2.COLOR_GRAY2BGR),
        "sharp": cv2.addWeighted(big, 1.6, cv2.GaussianBlur(big, (0, 0), 2), -0.6, 0),
    }
    if target_h > 0:                                   # extra variants only for plate crops, not the full photo
        out["large"] = prep(target_h * 1.6)
        g2 = cv2.cvtColor(prep(target_h * 0.7), cv2.COLOR_BGR2GRAY)
        out["clahe_small"] = cv2.cvtColor(cv2.createCLAHE(3.0, (8, 8)).apply(g2), cv2.COLOR_GRAY2BGR)
    return out

In [ ]:
# PATCH B - reader: valid plates of normal length (9-10 chars) get a small bonus over truncated ones
def read_plate_from_crop(crop, try_split=False):
    votes = defaultdict(float)

    def add(raw, conf, weight=1.0, whole=False, only_valid=False):
        plate, valid = best_plate(raw)
        if len(plate) < 4 or (only_valid and not valid):
            return
        w = conf * weight
        if valid:
            w *= 2.0 * (1.0 + 0.12 * (min(len(plate), 10) - 8))     # MH149239 (8) loses to MH14GN9239 (10)
        else:
            w *= 0.35 if whole else 0.5
        votes[(plate, valid)] += w

    for _, img in make_variants(crop).items():
        lines = ocr_lines(img)
        if lines:
            joined, conf = join_rows(lines)
            add(joined, conf, whole=True)
            for l in lines:
                add(l[3], l[4])
            big = sorted(lines, key=lambda l: -l[2])[:2]
            if len(big) == 2:
                add(join_rows(big)[0], conf, whole=True)

    if try_split or not any(v for (_, v) in votes):
        h = crop.shape[0]
        up = lambda im: cv2.resize(im, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        for a, b in [(0.55, 0.45), (0.50, 0.50), (0.60, 0.40)]:
            top, bot = crop[:int(h * a)], crop[int(h * b):]
            if top.size == 0 or bot.size == 0:
                continue
            t1, c1 = ocr_rec_only(up(top)); t2, c2 = ocr_rec_only(up(bot))
            if t1 and t2:
                add(t1 + t2, (c1 + c2) / 2, weight=0.8, only_valid=True)

    if not votes:
        return "", 0.0, False
    (plate, valid), score = max(votes.items(), key=lambda kv: (kv[0][1], kv[1]))
    return plate, round(min(score / 4.0, 1.0), 4), valid

In [ ]:
# PATCH C - stronger last-resort search for images where the detector finds nothing (car46 type)
def fallback_read(image):
    H, W = image.shape[:2]
    regions = [image, image[int(H * 0.35):]]                                  # whole photo, lower 65%
    for x0 in (0, int(0.2 * W), int(0.4 * W)):                                # 3 overlapping tiles, lower 70%
        regions.append(image[int(H * 0.30):, x0:x0 + int(0.6 * W)])
    found = defaultdict(float)
    for reg in regions:
        if reg.size == 0:
            continue
        s = float(np.clip(1400 / max(reg.shape[:2]), 1.0, 2.0))               # enlarge small regions up to 2x
        if s > 1.05:
            reg = cv2.resize(reg, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
        for _, img in make_variants(reg, target_h=0).items():
            lines = ocr_lines(img)
            for raw, conf in [(join_rows(lines)[0], 0.5)] + [(l[3], l[4]) for l in lines]:
                plate, valid = best_plate(raw)
                if valid:
                    found[plate] += conf
    return sorted(found.items(), key=lambda kv: -kv[1])[:MAX_PLATES_PER_IMAGE]


def process_image_v3(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return []
    name = os.path.basename(image_path); stem = os.path.splitext(name)[0]
    now = datetime.now(); d, t = now.strftime("%d-%m-%Y"), now.strftime("%H:%M:%S")
    rows, annotated = [], image.copy()

    for k, box in enumerate(detect_plates(image), start=1):
        crops, two_row_like = crop_options(image, box)
        best = None
        for crop in crops:
            if crop.size == 0:
                continue
            plate, conf, valid = read_plate_from_crop(crop, try_split=two_row_like)
            if plate and (best is None or (valid, conf) > (best[2], best[1])):
                best = (plate, conf, valid, crop)
            if valid and conf > 0.8:
                break
        if best:
            plate, conf, valid, crop = best
            cv2.imwrite(f"output/{stem}_plate{k}.jpg", crop)
            rows.append([name, plate, conf, valid, "detector", d, t])
            cv2.rectangle(annotated, box[:2], box[2:4], (0, 255, 0), 2)
            cv2.putText(annotated, plate, (box[0], max(20, box[1] - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    if not any(r[3] for r in rows):
        for plate, score in fallback_read(image):
            rows.append([name, plate, round(min(score / 3, 1.0), 4), True, "fallback", d, t])

    if any(r[3] for r in rows):
        rows = [r for r in rows if r[3]]
    cv2.imwrite(f"output/{stem}_annotated.jpg", annotated)
    return rows

In [ ]:
image_files = sorted(f for f in os.listdir(IMAGES_FOLDER)
                     if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")))
print(len(image_files), "images found")

all_rows, failed = [], []
t0 = time.time()
for i, file in enumerate(image_files, 1):
    try:
        rows = process_image_v3(os.path.join(IMAGES_FOLDER, file))
    except Exception as e:
        rows = []; print(f"ERROR {file}: {e}")
    (all_rows.extend(rows) if rows else failed.append(file))
    print(f"[{i:02d}/{len(image_files)}] {file:14s} -> {[r[1] for r in rows] or 'NOTHING FOUND'}")

df = pd.DataFrame(all_rows, columns=["Image Name", "Plate Number", "Confidence",
                                     "Valid Format", "Source", "Date", "Time"])
df.to_csv("database/plate_results.csv", index=False)
print(f"\nSaved {len(df)} rows to database/plate_results.csv in {time.time()-t0:.0f}s")
print("No result:", failed)

In [ ]:
# EVALUATION - same as before + a score for Indian-format plates only
gt = pd.read_csv(GROUND_TRUTH_CSV)
gt.columns = gt.columns.str.strip()
gt = gt.loc[:, ~gt.columns.str.contains("^Unnamed")]
gt = gt.replace(r"^\s*$", np.nan, regex=True).dropna(subset=["Image Name", "Actual Plate"]).reset_index(drop=True)
norm = lambda s: re.sub(r"[^A-Z0-9]", "", str(s).upper())
stem = lambda s: os.path.splitext(str(s).strip())[0]
gt["Image"], gt["Actual"] = gt["Image Name"].apply(stem), gt["Actual Plate"].apply(norm)

pred = pd.read_csv("database/plate_results.csv")
assert "Valid Format" in pred.columns, "Old CSV! Re-run the run cell first."
pred["Image"], pred["Pred"] = pred["Image Name"].apply(stem), pred["Plate Number"].apply(norm)
pred_map = pred.groupby("Image")["Pred"].apply(list).to_dict()

recs = []
for _, r in gt.iterrows():
    c = pred_map.get(r["Image"], [])
    best = max(c, key=lambda p: difflib.SequenceMatcher(None, r["Actual"], p).ratio()) if c else ""
    sim = difflib.SequenceMatcher(None, r["Actual"], best).ratio() * 100 if c else 0
    recs.append([r["Image"], r["Actual"], best, bool(c), r["Actual"] in c, round(sim, 1)])
res = pd.DataFrame(recs, columns=["Image", "Actual", "Best Prediction", "Detected", "Correct", "Similarity %"])
res["Indian format"] = res["Actual"].apply(lambda a: best_plate(a) == (a, True))

ind = res[res["Indian format"]]
print("=" * 55)
print(f"ALL ground-truth plates      : {len(res):3d}   exact accuracy {res['Correct'].mean()*100:6.2f}%")
print(f"Indian-format plates only    : {len(ind):3d}   exact accuracy {ind['Correct'].mean()*100:6.2f}%")
print(f"Non-standard / foreign plates: {len(res)-len(ind):3d}   (US-style, names, odd GT) - report separately")
print(f"Detection rate               : {res['Detected'].mean()*100:.2f}%")
print(f"Avg char similarity          : {res['Similarity %'].mean():.2f}%")
print("=" * 55)
display(res[~res["Correct"]])

In [ ]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "ultralytics",
    "paddleocr",
    "paddlepaddle",
    "opencv-python",
    "numpy",
    "pandas",
    "matplotlib",
    "jupyter"
]

for package in packages:
    try:
        print(f"{package}=={version(package)}")
    except PackageNotFoundError:
        print(f"{package}: Not installed under this package name")